# **Kernel Stage 开发指引**

本节是开发指引的第 2 节，重点说明 **Kernel 侧的阶段化代码组织**。在掌握了算子逻辑（02.06）、Win 内存布局（02.07）、Tiling 字段（02.08）之后，本节回答：

- Dispatch / Combine 的 `__global__` 入口长什么样？
- `Init` 与 `Process` 分别承担什么？
- 02.06 中讲的 1.1~1.4 / 2.1~2.2 阶段，落到代码里分别是哪个函数？
- 阶段之间用什么原语同步？

---

## **工程目录结构回顾**

本节涉及的代码文件与上一节（Tiling）相同，但视角从 Host 侧转向 Kernel 侧：

```
answer/scaffold_02_10/
├─ include/                       # ★ 本节重点：Kernel 算子实现
│  ├─ moe_distribute_dispatch.h           # Dispatch 算子类
│  │                                      #   - Init()：绑定 GM、计算派生量、分配 UB
│  │                                      #   - Process()：AllToAllDispatch / CalCumSum / LocalWindowCopy
│  │                                      #   - 各阶段子函数：TokenToExpert / CalAndSendCnt 等
│  ├─ moe_distribute_combine.h            # Combine 算子类
│  │                                      #   - Init()：绑定 GM、核分工切分
│  │                                      #   - Process()：四步顺序 BuffInit → Dispatch → AlltoAllBuffInit → LocalWindowCopy
│  │                                      #   - 自旋等待：WaitDispatch / ProcessMoeExpert
│  ├─ moe_distribute_dispatch_quant.h     # 量化模板（QuantInit/QuantProcess/CopyScalesToOut）
│  ├─ moe_distribute_dispatch_non_quant.h # 非量化版参考
│  └─ moe_distribute_comm.h               # Win 区偏移常量、aclshmemx 封装
├─ src/                           # ★ 本节重点：Kernel 入口
│  ├─ dispatch_and_combine_final.asc      # Kernel 入口函数
│  │                                      #   - DispatchKernel()：__global__ 入口
│  │                                      #   - CombineKernel()：__global__ 入口
│  │                                      #   - 栈构造 TPipe、栈构造算子类
│  ├─ 0_non_quant_naive.asc               # 非量化版入口（对比）
│  └─ utils.h                             # 测试辅助
└─ scripts/ cmake/                # 测试脚本与构建配置（上一节已熟悉）
```

**本节重点文件**：

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">文件</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">Kernel侧内容</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">本节关联</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>dispatch_and_combine_final.asc</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">DispatchKernel / CombineKernel 入口</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">★ 入口模板</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>moe_distribute_dispatch.h</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Dispatch算子类 Process()</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">★ 阶段拆解：AllToAllDispatch / CalCumSum / LocalWindowCopy</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>moe_distribute_combine.h</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Combine算子类 Process()</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">★ 阶段拆解：四步顺序</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>moe_distribute_comm.h</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Win偏移常量、aclshmemx_mte_put_nbi</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">同步原语、对端通信</td>
  </tr>
</table>

---

## **查看关键代码文件**

本节涉及的核心代码为 Kernel 入口和 Process 阶段函数：

In [ ]:
# 查看 Kernel 入口函数（DispatchKernel / CombineKernel）
!grep -A 20 '__global__.*DispatchKernel' scaffold_02_10/src/dispatch_and_combine_final.asc

In [ ]:
# 查看 Dispatch Process 阶段拆解
!grep -A 15 'void.*Process()' scaffold_02_10/include/moe_distribute_dispatch.h | head -20

In [ ]:
# 查看 Combine Process 阶段拆解
!grep -A 15 'void.*Process()' scaffold_02_10/include/moe_distribute_combine.h | head -20

In [ ]:
# 查看通信公共头（Win区偏移常量）
!cat scaffold_02_10/include/moe_distribute_comm.h | head -60

---
**两个 kernel 的整体框架对照**：

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">维度</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">Dispatch</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">Combine</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">入口函数</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>DispatchKernel</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>CombineKernel</code></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">算子类</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>DispatchImplNonQuant::MoeDistributeDispatch</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>MoeDistributeCombineShmemImpl::MoeDistributeCombineShmem</code></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">核分工策略</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>aivUsedAllToAll_</code> + <code>aivUsedCumSum_</code>（核间分组）</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">全核共同推进（核间按 token 切分）</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Process 阶段数</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">2 大阶段（分组 + 全核 LocalWindowCopy）</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">4 步顺序（BuffInit → 发 dispatch → AlltoAllBuffInit → LocalWindowCopy）</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Win 内存写入方式</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>aclshmemx_mte_put_nbi</code> 推数据 + 推状态块</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>aclshmemx_mte_put_nbi</code> 推数据 + 推 token-flag</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">自旋等待原语</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>Sum</code> 计算到达 token-flag 或 cnt 状态块</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>Sum</code> 计算 token 粒度 flag 是否到 K</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">跨阶段同步</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>PipeBarrier&lt;PIPE_ALL&gt;()</code> 在分组流程与 LocalWindowCopy 之间</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>PipeBarrier&lt;PIPE_ALL&gt;()</code> 在每两步之间</td>
  </tr>
</table>

---

## **1. Kernel 入口模板**

两个 kernel 入口遵循统一实现结构（节选自 [src/0_non_quant_naive.asc](./scaffold_01_10/src/dispatch_and_combine_final.asc)）：

```cpp
__global__ __aicore__ __vector__ void DispatchKernel(
    GM_ADDR shmemSpace, GM_ADDR x, GM_ADDR expertIds,
    GM_ADDR expandXOut, GM_ADDR dynamicScalesOut, GM_ADDR assistInfoOut, GM_ADDR expertTokenNumsOut, GM_ADDR epSendCountsOut,
    GM_ADDR workspaceGM, DispatchTilingData tilingData)
{
    AscendC::TPipe pipe;
    DispatchImplNonQuant::MoeDistributeDispatch dispatchOp;
    dispatchOp.Init(shmemSpace, x, expertIds,
        expandXOut, assistInfoOut, expertTokenNumsOut, epSendCountsOut,
        workspaceGM, &pipe, &tilingData);
    dispatchOp.Process();
    return;
}

__global__ __aicore__ __vector__ void CombineKernel(
    __gm__ void* shmemSpace, GM_ADDR expandX, GM_ADDR expertIds, GM_ADDR expandIdx,
    GM_ADDR epSendCount, GM_ADDR expertScales, GM_ADDR XOut,
    GM_ADDR workspaceGM, MoeDistributeCombineShmemTilingData combineTilingData)
{
    AscendC::TPipe pipe;
    MoeDistributeCombineShmemImpl::MoeDistributeCombineShmem<float16_t, float16_t, int32_t> op;
    op.Init((GM_ADDR)shmemSpace, expandX, expertIds, expandIdx, epSendCount, expertScales, XOut, workspaceGM, &pipe, combineTilingData);
    op.Process();
    return;
}
```

---

## **2. 入口核心逻辑**

1. **栈上构造 `TPipe`**：所有 UB 分配、队列、缓冲都注册到这个 `TPipe`；离开 kernel 时自动析构。
2. **栈上构造算子类对象**：Dispatch / Combine 实现写成模板/普通类，所有跨阶段状态变量（如 `aivId_`、`dataState_`、`winDataSizeOffset_`、`hCommuSize_`）作为类成员存放，避免函数间反复传参。
3. **调 `Init` 再调 `Process`**：`Init` 负责一次性初始化（绑 GlobalBuffer、读 TilingData、算派生量、设 `dataState_`、分配 UB 缓冲），`Process` 是真正的多阶段流水。

**Init 的子步骤（以 Dispatch 为例）**：

```cpp
void Init(...) {
    SetCtrlSpr<FLOAT_OVERFLOW_MODE_CTRL, FLOAT_OVERFLOW_MODE_CTRL>(0);  // 浮点溢出模式
    tpipe_ = pipe;
    aivId_ = GetBlockIdx();              // 本核 id
    totalWinSize_ = tilingData->totalWinSizeEp;
    shmemContextGM_ = shmemSpace;
    xGMTensor_.SetGlobalBuffer(...);     // 绑各 GlobalBuffer
    // ...
    SetTilingDataAndCal(tilingData);     // 拷字段 + 算派生量 hCommuSize_、aivUsedCumSum_、...
    SetDataStatus();                     // 读 dataState_、写翻转，定 winDataSizeOffset_、winStatusOffset_
    MaxSizeCal();                        // 估算 UB 上下文，准备 BufferInit
}
```

**实现说明**：`TPipe` 是 **kernel 入口在栈上 new** 的，而不是 kernel 类的成员；Init 拿到的是它的指针，整个 Process 期间复用。不能在 Process 内部再 new 第二个 `TPipe`。

---

## **3. Dispatch · Process 阶段拆解**

Dispatch 的 `Process` 函数极短：

```cpp
void MoeDistributeDispatch::Process() {
    if ASCEND_IS_AIV {
        if (aivId_ < aivUsedAllToAll_) {
            AllToAllDispatch();   // 前段核
        } else {
            CalCumSum();          // 后段核
        }
        PipeBarrier<PIPE_ALL>();  // 全核屏障
        LocalWindowCopy();        // 全核共同执行
    }
}
```

**核分工**：`aivUsedAllToAll_ = aivNum_ - aivUsedCumSum_`。前 `aivUsedAllToAll_` 个核走「主分发」，剩下的核（最多 `CUMSUM_MAX_CORE_NUM = 16`）走「核间软同步 + 前缀和」。两组并行，最后通过 `PipeBarrier<PIPE_ALL>()` 汇合再共同进入 `LocalWindowCopy`。

**与 02.06 阶段编号的对应**：

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">02.06 阶段</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">函数</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">调用方</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">关键动作</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">1.1 token 分发</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>AllToAllDispatch</code> → <code>SendToMoeExpert</code> → <code>TokenToExpert</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">前段核</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">把每条 token 按 topk 推送到目标卡 Win数据区，用 <code>aclshmemx_mte_put_nbi</code></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">1.2 状态块发送</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>AllToAllDispatch</code> → <code>CalAndSendCnt</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">前段核</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">在 1.1 之后给每对 (源卡, 本地专家) 推一个 32B 状态块（flag + count）</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">1.3 接收侧软同步 + 前缀和</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>CalCumSum</code> → <code>GatherSumRecvCnt</code> → <code>WaitCumSumFlag</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">后段核</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">自旋等状态块汇齐 → 算各组 token 累计偏移 → 置位 <code>CUMSUM_FLAG_OFFSET</code></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">1.4 等到达 + 整理</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>LocalWindowCopy</code> → <code>CheckDataArriveWithFlag</code> → <code>CopyInAndOut</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">全核</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">用 token-flag 自旋（<code>Sum(blockCntPerToken_ × 32B flag)</code>），到齐后从本卡 Win 搬到 <code>expandXOut</code></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">1.4 reset</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>LocalWindowCopy</code> 内部</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">全核</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">整理过程中同步把已读 token-flag / 状态块写 0</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">1.4 输出 sendCounts / expertTokenNums</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>WaitAndFormatOutput</code> + <code>SetExpertTokenNums</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">全核</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">把 cumsum 结果格式化成 <code>sendCountsOut</code> 与 <code>expertTokenNumsOut</code></td>
  </tr>
</table>

**实现说明**：

- 1.4 之所以放在 `PipeBarrier<PIPE_ALL>` 之后全核共同执行，是因为它依赖 1.3 的前缀和结果（决定本核处理哪几条 token）以及 1.1/1.2 写入的全部 Win 数据；屏障保证两组核都完成。
- `LocalWindowCopy` **内部包含 reset 写 0**，所以即便本卡数据已到齐，也必须先经过屏障——否则后段核还在写状态块，前段核已经把状态块清零会导致丢同步。源码注释：`// localWindowCopy中包含reset操作，需确保前面操作完成`。

**核内自旋示意（CheckDataArriveWithFlag）**：

```cpp
// 把 blockCntPerToken_ 个 token-flag 32B 块求和，到 1.0*blockCntPerToken 即代表到齐
while (Sum(flagBuf) < expected) { /* 重试 */ }
```

---

## **4. Combine · Process 阶段拆解**

Combine 的 `Process` 是四步顺序：

```cpp
void MoeDistributeCombineShmem::Process() {
    if ASCEND_IS_AIV {
        BuffInit();
        SetWaitStatusAndDisPatch();   // 内含 ExpertAlltoAllDispatchCopyAdd
        PipeBarrier<PIPE_ALL>();
        AlltoAllBuffInit();
        LocalWindowCopy();
    }
}
```

**核分工**：所有 AIV 核共同推进每一步，仅核内按 token 范围切分（`SplitCoreCal` 在 Init 内已算好 `startTokenId_ / endTokenId_`）。没有 dispatch 那种「前段/后段」的核分组。

**与 02.06 阶段编号的对应**：

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">02.06 阶段</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">函数</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">关键动作</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">2.1 推回送 token + token-flag</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>SetWaitStatusAndDisPatch</code> → <code>ExpertAlltoAllDispatchCopyAdd</code> → <code>ExpertAlltoAllDispatchInnerCopyAdd</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">把本卡持有的「专家×topk×token」回送给原 token-owner 卡：先 <code>aclshmemx_mte_put_nbi</code> 推 token 到对端 Win数据区，再推一个 32B token-flag 到对端 Win状态区</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">2.2 等 K 路 token-flag 全到</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>LocalWindowCopy</code> → <code>WaitDispatch</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">对本卡每条 token，<code>Sum</code> 求 <code>flagRcvCount_ = K</code> 个 token-flag，到 K 即到齐；到齐后同步 reset 写 0</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">2.2 加权汇总 K 路</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>LocalWindowCopy</code> → <code>ProcessMoeExpert</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">K 路 token DataCopy 到 UB → <code>Cast</code> 到 fp32 → <code>Muls(scale)</code> → 累加到 <code>sumFloatBufLocal_</code> → <code>Cast</code> 回 fp16 → <code>DataCopyPad</code> 到 <code>XOut</code></td>
  </tr>
</table>

**两步之间的 PipeBarrier**：在 `SetWaitStatusAndDisPatch` 和 `AlltoAllBuffInit` 之间需要 `PipeBarrier<PIPE_ALL>`，因为 `AlltoAllBuffInit` 会 `tpipe_->Reset()` 清空 UB 缓冲，必须等所有核完成第一阶段写入才能安全 reset。源码注释：`// AlltoAllBuffInitAndMaskCal中包含reset操作，需确保前面操作完成`。

**WaitDispatch 自旋的实现**：

```cpp
uint32_t copyCount = flagRcvCount_ * FLOAT_PER_UB_ALIGN;   // K × 8 个 float
float target = 1.0f * copyCount;                            // 期望和 = copyCount
while (localState 不在 [target-0.5, target+0.5]) {
    DataCopy(stateTensor, stateGMTensor, copyCount);        // 读 Win状态区
    Sum(stateTensor, stateTensor, ...);                     // 求和
    localState = stateTensor(0);
}
DataCopy(stateGMTensor, stateResetTensor_, copyCount);      // 清零
```

**实现说明**：

- Combine 与 Dispatch 在自旋上策略一致（都是 UB-side `Sum` 比较），但**flag 形态不同**：Dispatch 的 token-flag 嵌入数据流（480 + 32），Combine 的 token-flag 单独放在 Win状态区（每条 token K 个 32B 槽位）。
- `BuffInit` 与 `AlltoAllBuffInit` 是两套不同的 UB 布局：第一步偏向「scale 计算 + send」，第二步偏向「Sum 汇总 + 输出」。Combine 在两步间显式 `tpipe_->Reset()` 切换布局，是为了在 UB 有限的情况下复用同一份内存。

---

## **5. 关键工具与同步原语清单**

**对端通信**（来自 `aclshmemx`）：

- `aclshmemx_mte_put_nbi<T>(dstGlobalTensor, srcLocalTensor, count, peerRankId, eventId)`：非阻塞推送 UB → 对端 GM。Dispatch / Combine 推 token 数据、推 token-flag、推状态块都用这一个原语。
- `aclshmem_ptr(shmemSpace, peerRankId)`：拿到对端 pe 的Win 区基址；与 `STATUS_REGION_OFFSET` 配合区分Win数据区 / Win状态区。
- `aclshmem_malloc / aclshmem_align / aclshmem_free`：host 端Win 区内存分配。

**核内同步**（AscendC 原语）：

- `PipeBarrier<PIPE_ALL>()`：屏障当前核所有流水（MTE2/MTE3/V/S）；用于跨阶段同步、reset 前的安全点。
- `SyncFunc<AscendC::HardEvent::XXX>()`：单核内两段流水之间的事件同步（如 `MTE2_V`、`V_MTE3`、`S_MTE3`）；用于 DataCopy → Vector 计算 → DataCopy 链路。

**核间软同步**（自旋实现，无硬件原语）：

- Dispatch 的 `WaitCumSumFlag` 和 Combine 的 `WaitDispatch` 都是通过轮询状态区、`Sum` 后比较阈值的方式实现。
- 这类软同步依赖“写入顺序 + 清零顺序”，所以前后阶段之间都要有 `PipeBarrier<PIPE_ALL>` 作为安全点。

**小结**：如果记不住具体实现，先记住三个层次：对端通信负责搬数据，`PipeBarrier` 负责阶段切换，`Sum + while` 负责等状态。

---

## **6. 实操要点小结**

- **入口保持精简**：`__global__` 函数仅执行三项核心逻辑：栈构造 `TPipe`、栈构造算子类、调用 `Init` 与 `Process`。业务逻辑应放在算子类成员函数中，避免在入口函数中引入计算细节或传递裸指针成员。
- **Init 完成所有派生量计算**：所有依赖编译期常量的派生（`hCommuSize_`、`aivUsedCumSum_`、`statusCntAlign_` 等）一次性算清并写入成员变量，`Process` 期间不再重算；这能避免「同一公式出现在两个地方而值不一致」的隐 bug。
- **Process 严守阶段顺序**：Dispatch 的 1.x、Combine 的 2.x 阶段之间必须有 `PipeBarrier<PIPE_ALL>()` 屏障；尤其是「下一阶段会 reset 或读对端写入的状态」时，缺失屏障极易导致偶发死锁或数据错。
- **核分工策略不同要区分**：Dispatch 是「前段/后段两组核并行 → 屏障合并 → 全核 LocalWindowCopy」，Combine 是「全核共同推进 4 步」。理解这一差异能解释为什么 dispatch 需要 `aivUsedAllToAll_ / aivUsedCumSum_` 两个值而 combine 不需要。
- **自旋 = Sum，不是裸 while 读单字**：Win状态区的 flag 总以 32 B（8 × fp32 「1.0」）为粒度，自旋的统一形态是「DataCopy 到 UB → Sum → 比较」。读单一字节判等会撞硬件一致性边界、且没法摊销 DMA 延迟。
- **token-flag 形态因阶段不同**：Dispatch 1.4 的 token-flag 嵌入数据流（480 + 32），Combine 2.2 的 token-flag 在状态区独立槽位（每 token K 个 32 B）；写自旋时要先想清楚当前阶段读的是哪种 flag、布在哪个偏移。

---

**导航**：[← 上一章：Tiling开发指引](02.08_tiling_guide.ipynb) | [返回课程主页](02.01_chapter_intro.ipynb) | [下一章：量化实践 →](02.10_quant_dispatch_practice.ipynb)

> **下一步**：完成理论学习后，进入 [02.10 量化实践](02.10_quant_dispatch_practice.ipynb) 进行实际代码开发
---